# Neuro-Symbolic Agent Architecture | Cognitive Architectures

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from typing_extensions import NotRequired
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke
import json
import re

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
# Symbolic module: deterministic tax rules (no LLM involved)
TAX_BRACKETS_2024 = [
    (11600, 0.10), (47150, 0.12), (100525, 0.22),
    (191950, 0.24), (243725, 0.32), (609350, 0.35), (float("inf"), 0.37),
]

def calculate_tax(income: float) -> dict:
    """Symbolic tax calculation -- deterministic, no LLM."""
    if income <= 0:
        return {"total_tax": 0, "effective_rate": 0, "breakdown": []}
    tax = 0.0
    prev_limit = 0
    breakdown = []
    for limit, rate in TAX_BRACKETS_2024:
        if income <= prev_limit:
            break
        taxable = min(income, limit) - prev_limit
        bracket_tax = taxable * rate
        tax += bracket_tax
        breakdown.append({"bracket": f"{rate:.0%}", "taxable": taxable, "tax": bracket_tax})
        prev_limit = limit
    return {"total_tax": round(tax, 2), "effective_rate": round(tax / income * 100, 2), "breakdown": breakdown}

class NeuroSymState(TypedDict):
    user_query: str
    parsed_income: NotRequired[float]
    tax_result: NotRequired[dict]
    response: NotRequired[str]

def neural_parse(state: NeuroSymState) -> dict:
    """Neural module: extract structured data from natural language."""
    response = model.invoke(
        f"Extract the annual income from this query. Return ONLY a JSON number.\n\n"
        f"Query: {state['user_query']}\n\nReturn: {{\"income\": 75000}}"
    )
    cleaned = re.sub(r"```(?:json)?\s*|\s*```", "", response.content).strip()
    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed, (int, float)):
            income = float(parsed)
        else:
            income = float(parsed.get("income", 0))
    except (json.JSONDecodeError, TypeError, ValueError, AttributeError):
        income = 0
    return {"parsed_income": income}

def symbolic_calculate(state: NeuroSymState) -> dict:
    """Symbolic module: precise tax calculation."""
    result = calculate_tax(state["parsed_income"])
    return {"tax_result": result}

def neural_respond(state: NeuroSymState) -> dict:
    """Neural module: format the result as a friendly response."""
    response = model.invoke(
        f"Format this tax calculation as a friendly, clear response:\n\n"
        f"User asked: {state['user_query']}\n"
        f"Income: ${state['parsed_income']:,.2f}\n"
        f"Tax result: {json.dumps(state['tax_result'], indent=2)}\n\n"
        f"Explain the brackets and effective rate clearly."
    )
    return {"response": response.content}

In [5]:
graph = StateGraph(NeuroSymState)
graph.add_node("parse", neural_parse)
graph.add_node("calculate", symbolic_calculate)
graph.add_node("respond", neural_respond)
graph.add_edge(START, "parse")
graph.add_edge("parse", "calculate")
graph.add_edge("calculate", "respond")
graph.add_edge("respond", END)

agent = graph.compile()

In [6]:
# Plot the workflow
plot_mermaid(agent)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	parse(parse)
	calculate(calculate)
	respond(respond)
	__end__([<p>__end__</p>]):::last
	__start__ --> parse;
	calculate --> respond;
	parse --> calculate;
	respond --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [7]:
result = agent.invoke({"user_query": "I make $85,000 a year. How much federal tax do I owe?"})
print(result["response"])

Hello! I'd be happy to help you understand how much federal tax you owe on an income of $85,000 and break down the details for you.

### Total Federal Tax Owed
- **$13,753**

### Effective Tax Rate
- **16.18%**: This rate represents the portion of your total income that goes to federal taxes. It's calculated by dividing your total tax ($13,753) by your total income ($85,000), and then multiplying by 100 to get a percentage. 

### Breakdown by Tax Bracket:
Your income is taxed at different rates as it falls within specific brackets. Here's how it breaks down:

1. **10% Bracket**:
   - **Taxable Income**: $11,600
   - **Tax**: $1,160
   - The first $11,600 of your income is taxed at 10%.

2. **12% Bracket**:
   - **Taxable Income**: $35,550
   - **Tax**: $4,266
   - The next portion of your income, from $11,600 to $47,150, is taxed at 12%.

3. **22% Bracket**:
   - **Taxable Income**: $37,850
   - **Tax**: $8,327
   - Finally, the portion of your income over $47,150 up to your income of 

In [8]:
stream_invoke(agent, {"user_query": "I make $85,000 a year. How much federal tax do I owe?"})


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'user_query': 'I make $85,000 a year. How much federal tax do I owe?',
 'parsed_income': 85000.0,
 'tax_result': {'total_tax': 13753.0,
  'effective_rate': 16.18,
  'breakdown': [{'bracket': '10%', 'taxable': 11600, 'tax': 1160.0},
   {'bracket': '12%', 'taxable': 35550, 'tax': 4266.0},
   {'bracket': '22%', 'taxable': 37850.0, 'tax': 8327.0}]},
 'response': "Hello! I'd be happy to help you understand how much federal tax you owe on an income of $85,000 and break down the details for you.\n\n### Total Federal Tax Owed\n- **$13,753**\n\n### Effective Tax Rate\n- **16.18%**: This rate represents the portion of your total income that goes to federal taxes. It's calculated by dividing your total tax ($13,753) by your total income ($85,000), and then multiplying by 100 to get a percentage. \n\n### Breakdown by Tax Bracket:\nYour income is taxed at different rates as it falls within specific brackets. Here's how it breaks down:\n\n1. **10% Bracket**:\n   - **Taxable Income**: $11,600\n   - 